In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from sympy import latex

In [ ]:
#Function that integrate a symbolic function ff on the variable x on R using the standard Gaussian measure
def esperance(ff, x):
  
#    ff : expression sympy
#        Fonction à intégrer
#    x : symbol sympy        

    return sp.integrate(ff*sp.exp(-x*x/2)/(sp.sqrt(2*sp.pi)), (x, -sp.oo, sp.oo))

In [ ]:
# Symbolic variables definition
x1, x2 = sp.symbols('x1 x2')
ro = sp.symbols('ro')

In [ ]:
# Defining a simple regression function 
roortho=sp.sqrt(1-ro*ro)
f = ro*x1+roortho*x2
f2=f*f

In [ ]:
# Computing the conditional expectation of f with respect to X1=x1
esperance_condf=esperance(f,x2)
# Computing the conditional expectation of f² with respect to X1=x1
esperance_condf2=esperance(f2,x2)

In [ ]:
#Computing Sigma0
#Building first the vector(esperance_condf)^2 esperance_condf espeance_condf² 
vbric_Sigma0 = sp.Matrix([esperance_condf*esperance_condf, esperance_condf, esperance_condf2])
#Computing the centering part
espvbric_Sigma0=esperance(vbric_Sigma0,x1)
centrageSigma0=espvbric_Sigma0*espvbric_Sigma0.T
#Computing the quadratic part before integrating with respect to x1
mbric_Sigma0=vbric_Sigma0*vbric_Sigma0.T
#Computing the covariance matrix Sigma0
Sigma0=esperance(mbric_Sigma0, x1)-centrageSigma0

In [ ]:
#Computing Sigma_a
vbric_Sigma_a = sp.Matrix([2*esperance_condf, 1, 1])
unzeze=sp.Matrix([1, 0, 0])
matrix_de_1_et_0=unzeze*unzeze.T
covepsf=esperance_condf2-esperance_condf*esperance_condf
Sigma_a=vbric_Sigma_a*vbric_Sigma_a.T+covepsf*matrix_de_1_et_0

In [ ]:
#Computing Sigma_b
vbric_Sigma_b= sp.Matrix([f, f, f2])
#Computing the conditional expectation of the last vector
espvbric_Sigma_b=sp.Matrix([esperance_condf, esperance_condf, esperance_condf2])
#Computing Sigma_b
mbric_Sigma_b=vbric_Sigma_b*vbric_Sigma_b.T
Sigma_b=esperance(mbric_Sigma_b,x2)-espvbric_Sigma_b*espvbric_Sigma_b.T

In [ ]:
#Computing Sigma1
# Computing the integrand Hadamard product of Sigma_a and Sigma_b
Igrand_Sigma1=sp.hadamard_product(Sigma_a,Sigma_b)
#Finally integrating with respect to x1
Sigma1=esperance(Igrand_Sigma1, x1)

In [ ]:
#Computuation of Sobol' index for X1 and the variance of f
#Expectation of f
espf=esperance(esperance_condf,x1)
#Expectation of f²
espf2=esperance(esperance_condf2,x1)
#Variance of f
varY=espf2-espf*espf
#Computation of the variance of the conditional expectation
numerator_Zobol=esperance(esperance_condf*esperance_condf,x1)-espf*espf
#Computing Sobol' index
Zobol=sp.simplify(numerator_Zobol/varY)
print(latex(sp.simplify(Zobol)))

In [ ]:
#Computing the asymptotique variance of Sobol' index  Chatterjee estimator
v_f=(sp.Matrix([1, 2*espf*(Zobol-1), -Zobol]))/varY
#Asymptotic variance final formula
sigma_Zobol_hat=sp.simplify(v_f.T*(Sigma0+Sigma1)*v_f)
print(latex(sp.simplify(sigma_Zobol_hat)))

In [ ]:
#Evauating numerically Sobol' index and the asymptotic variance for special value of a and for a serie of b
#Transforming in numerical functions
Zobol_num = sp.lambdify(ro, Zobol, "numpy")
sigma_Zobol_hat_num = sp.lambdify(ro, sigma_Zobol_hat, "numpy")
# ro moves from -1 to 1
ro_vals = np.arange(-1, 1.01, 0.01)
Zobol_vals = Zobol_num(ro_vals)
sigma_Zobol_hat_vals = sigma_Zobol_hat_num(ro_vals).reshape(-1)
var_est=np.loadtxt('var_est_gauss.dat');
Sestim=np.loadtxt('S1_est_Chat_gauss.dat');


In [ ]:
# Create two plot asided
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ----- first plot -----
axes[0].plot(ro_vals,Zobol_vals,ro_vals,Sestim)
axes[0].set_title("Sobol' index in the Gaussian model")
axes[0].set_xlabel(r"$\rho$")
axes[0].set_ylabel("Sobol' index and its estimate")
axes[0].grid(True)
# ----- second plot -----
axes[1].plot(ro_vals,sigma_Zobol_hat_vals,ro_vals,var_est)
axes[1].set_title("Variance of the Chatterjee estimator")
axes[1].set_xlabel(r"$\rho$")
axes[1].set_ylabel("Asymptotic variance and its estimate")
axes[1].grid(True)

# Ajusting spaces automatically
#plt.tight_layout()

#saving the plot
plt.savefig("graph_Sob_Gaussian.png", dpi=300)
#Display
plt.show()